# SARF AraBERT — E3

**Owner:** A  
**Condition:** E3  
**CPT assignment:** general_12k_final_v2.csv (12,000 rows).

CPT on the approved 12K general Saudi-style corpus, then identical MSA fine-tuning.

This notebook contains no model-training implementation. It invokes the one shared `arabert_pipeline.py`, which reads the one shared AraBERT config and the one global 77-label mapping from the project Drive. The held-out Saudi test is not loaded in this notebook.


## 1. Install the fixed environment

Choose a GPU runtime before running this cell. If Colab asks to restart after installation, restart the runtime, then rerun this cell and the following cells in order.


In [1]:
%pip install --upgrade --prefer-binary "tokenizers==0.21.4" "transformers==4.48.3" "accelerate==1.2.1" "datasets==3.2.0" "scikit-learn==1.6.1" "sentencepiece>=0.2.1"
%pip install --upgrade "PyArabic==0.6.15" "farasapy==0.1.1" "emoji==1.4.2"
%pip install --upgrade --no-deps "arabert==1.0.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Success

## 2. Mount Drive and verify the shared AraBERT source

The project Drive must contain exactly one shared pipeline, config, and label mapping. This cell does not load training data, CPT data, or the held-out test.


In [2]:
from google.colab import drive
from pathlib import Path
import json
import torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT")
PIPELINE_PATH = PROJECT_ROOT / "src_04" / "arabert_pipeline.py"
CONFIG_PATH = PROJECT_ROOT / "02_configs" / "arabert_config.json"
MAPPING_PATH = PROJECT_ROOT / "02_configs" / "arabert_label_mapping.json"

assert torch.cuda.is_available(), "GPU required; do not run AraBERT training on CPU."
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"
assert PIPELINE_PATH.is_file(), f"Pipeline not found: {PIPELINE_PATH}"
assert CONFIG_PATH.is_file(), f"Config not found: {CONFIG_PATH}"
assert MAPPING_PATH.is_file(), f"Mapping not found: {MAPPING_PATH}"

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
mapping = json.loads(MAPPING_PATH.read_text(encoding="utf-8"))

assert config["owner"] == "A"
assert config["official_conditions"] == ["E0", "E1", "E2", "E3", "EB"]
assert config["official_seeds"] == [42, 123, 2026]
assert mapping["owner"] == "A"
assert mapping["num_labels"] == 77
assert len(mapping["label_to_id"]) == 77

print({
    "status": "shared AraBERT source verified",
    "owner": config["owner"],
    "condition": "E3",
    "gpu": torch.cuda.get_device_name(0),
    "pipeline": str(PIPELINE_PATH),
    "config": str(CONFIG_PATH),
    "mapping": str(MAPPING_PATH),
    "official_seeds": config["official_seeds"],
    "num_labels": mapping["num_labels"],
    "cpt_assignment": config["conditions"]["E3"],
})


Mounted at /content/drive
{'status': 'shared AraBERT source verified', 'owner': 'A', 'condition': 'E3', 'gpu': 'Tesla T4', 'pipeline': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py', 'config': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_config.json', 'mapping': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_label_mapping.json', 'official_seeds': [42, 123, 2026], 'num_labels': 77, 'cpt_assignment': {'description': 'CPT on 12K general Saudi-style text, then identical MSA fine-tuning.', 'cpt_enabled': True, 'cpt': {'corpus_directory_relative_path': '03_synthetic_data/05_final_corpora', 'manifest_relative_path': '03_synthetic_data/06_final_audit/corpus_manifest_v2.json', 'corpus_files': ['general_12k_final_v2.csv'], 'corpus_file_rows': [12000], 'total_rows': 12000}}}


## 3. Run E3 one seed at a time

Run only one seed per execution. The approved sequence within this condition is `42`, then `123`, then `2026`. The shared pipeline creates outputs directly under `05_runs/arabert/E3/seed_<seed>` and `06_models_and_checkpoints/arabert/E3/seed_<seed>`, refusing to overwrite a completed run.


In [3]:
SEED = 42
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E3" --seed {SEED}


2026-09-12 12:38:50.107885: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 611/611 [00:00<00:00, 4.61MB/s]
config.json: 100% 384/384 [00:00<00:00, 3.00MB/s]
vocab.txt: 720kB [00:00, 8.55MB/s]
tokenizer.json: 2.31MB [00:00, 47.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 1.14MB/s]
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100% 241M/241M [06:38<00:00, 606kiB/s]
[2026-09-12 12:45:42,184 - farasapy_logger - WARNING]: Be careful with large l

In [4]:
SEED = 123
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E3" --seed {SEED}


2026-09-12 13:09:50.580244: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-12 13:09:57,709 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:00<00:00, 12395.74 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 12174.86 examples/s]
Tokenizing approved E3 CPT corpus: 100% 12000/12000 [00:00<00:00, 13138.89 examples/s]
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and 

In [5]:
SEED = 2026
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E3" --seed {SEED}


2026-09-12 13:27:48.948552: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-12 13:27:55,949 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:01<00:00, 8700.52 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 7312.27 examples/s]
Tokenizing approved E3 CPT corpus: 100% 12000/12000 [00:01<00:00, 8446.45 examples/s]
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and oth